In [ ]:
# Cell 1: Fetch Forecast (next 7 days)
# =====================================================

import sys
sys.path.append('../src')

from fetch_weather import fetch_forecast
import warnings
warnings.filterwarnings('ignore')

# Fetch today's forecast (saves as weather_forecast_YYYY-MM-DD.csv)
df_forecast = fetch_forecast(save_versioned=True)

In [ ]:
# Cell 2: Fetch Historical Actuals
# =====================================================

import sys
sys.path.append('../src')

from fetch_weather import fetch_historical
import warnings
warnings.filterwarnings('ignore')


# Backfill entire years (for initial setup)
# df_historical = fetch_historical("2020-01-01", "2099-12-31")


# Automatically pick up only missing days
df_historical = fetch_historical(auto_fill=True)



In [ ]:
# Cell 3: Process & Clean Data
# =====================================================

import sys
sys.path.append('../src')

from clean_weather import process_historical_hourly, load_raw_file, append_to_history, RAW_PATH
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Suppress verbose output from processing
from io import StringIO
from contextlib import redirect_stdout

# Process Historical Actuals (Hourly)
try:
    with redirect_stdout(StringIO()):
        hourly_actuals = process_historical_hourly()
        history_actuals = append_to_history(hourly_actuals, history_type="actuals", granularity="hourly")
except FileNotFoundError as e:
    pass

# Process Latest Forecast (Hourly)
try:
    forecast_files = sorted(RAW_PATH.glob("weather_forecast_*.csv"), reverse=True)
    if forecast_files:
        with redirect_stdout(StringIO()):
            raw = load_raw_file(forecast_files[0])
            
            # Add helper columns
            raw["date"] = pd.to_datetime(raw["datetime"]).dt.date
            raw["hour"] = pd.to_datetime(raw["datetime"]).dt.hour
            raw["year"] = pd.to_datetime(raw["datetime"]).dt.year
            raw["month"] = pd.to_datetime(raw["datetime"]).dt.month
            raw["month_name"] = pd.to_datetime(raw["datetime"]).dt.strftime("%B")
            raw["day_of_week"] = pd.to_datetime(raw["datetime"]).dt.strftime("%A")
            
            history_forecast = append_to_history(raw, history_type="forecasts", granularity="hourly")
except Exception as e:
    pass

print ("Complete")